In [1]:
import time
import random
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset

from torchvision import datasets, transforms, models

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cpu


In [3]:
IMG_SIZE = 128          # smaller than 224 -> faster
BATCH_SIZE = 128        # increase on GPU; reduce if OOM
NUM_WORKERS = 2
PIN_MEMORY = True if DEVICE == "cuda" else False

TRAIN_SUBSET = 6000     # use subset for fast demo
VAL_SUBSET   = 1000

EPOCHS_TL_HEAD = 2      # train classifier head only
EPOCHS_FT      = 2      # fine-tune last block
EPOCHS_SCRATCH = 2

In [4]:
train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [5]:
train_full = datasets.CIFAR10(root=".", train=True, download=True, transform=train_tfms)

100%|██████████| 170M/170M [31:02<00:00, 91.5kB/s]   


In [6]:
val_full   = datasets.CIFAR10(root=".", train=False, download=True, transform=val_tfms)

In [7]:
# Choose random subset indices
train_idx = random.sample(range(len(train_full)), TRAIN_SUBSET)
val_idx   = random.sample(range(len(val_full)), VAL_SUBSET)

train_ds = Subset(train_full, train_idx)
val_ds   = Subset(val_full, val_idx)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)

num_classes = 10
class_names = train_full.classes
print("Classes:", class_names)
print(f"Train subset: {len(train_ds)} | Val subset: {len(val_ds)}")

# --------------------

Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Train subset: 6000 | Val subset: 1000


In [8]:
def accuracy_from_logits(logits, y):
  preds = logits.argmax(dim=1)
  return (preds == y).float().mean().item()

In [9]:
def run_epoch(model, loader, criterion, optimizer=None):
    """
    If optimizer is provided -> train mode, else eval mode.
    Returns: avg_loss, avg_acc
    """
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    total_correct = 0
    total = 0

    with torch.set_grad_enabled(is_train):
        for x, y in loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            if is_train:
                optimizer.zero_grad()

            logits = model(x)
            loss = criterion(logits, y)

            if is_train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * x.size(0)
            total_correct += (logits.argmax(1) == y).sum().item()
            total += x.size(0)

    return total_loss / total, total_correct / total

In [10]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs, label=""):
    start = time.time()
    history = []
    for ep in range(1, epochs + 1):
        tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer=optimizer)
        va_loss, va_acc = run_epoch(model, val_loader, criterion, optimizer=None)
        history.append((tr_loss, tr_acc, va_loss, va_acc))
        print(f"{label}Epoch {ep:02d}/{epochs} | "
              f"train acc {tr_acc:.3f} loss {tr_loss:.4f} | "
              f"val acc {va_acc:.3f} loss {va_loss:.4f}")
    elapsed = time.time() - start
    return history, elapsed

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [11]:
def build_mobilenetv2_transfer(num_classes=10):
    model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

    # Freeze feature extractor
    for p in model.features.parameters():
        p.requires_grad = False

    # Replace classifier
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model

In [12]:
print("\n==============================")
print("A) Transfer Learning: Train head only (frozen backbone)")
print("==============================")

model_tl = build_mobilenetv2_transfer(num_classes).to(DEVICE)
print("Trainable params (head-only):", count_trainable_params(model_tl))


A) Transfer Learning: Train head only (frozen backbone)
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to C:\Users\liyoubao/.cache\torch\hub\checkpoints\mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 53.7MB/s]


Trainable params (head-only): 12810


In [13]:
criterion = nn.CrossEntropyLoss()
optimizer_head = optim.Adam(model_tl.classifier.parameters(), lr=1e-3)

hist_tl, time_tl = train_model(
    model_tl, train_loader, val_loader,
    criterion, optimizer_head,
    epochs=EPOCHS_TL_HEAD,
    label="[TL-Head] "
)
print(f"[TL-Head] Training time: {time_tl:.1f}s")

[TL-Head] Epoch 01/2 | train acc 0.604 loss 1.4344 | val acc 0.728 loss 0.9614
[TL-Head] Epoch 02/2 | train acc 0.744 loss 0.8601 | val acc 0.756 loss 0.8145
[TL-Head] Training time: 203.7s


In [14]:
# 7) Task B: Fine-tuning (unfreeze last layers)
# -----------------------------
print("\n==============================")
print("B) Fine-tuning: Unfreeze last conv blocks + classifier")
print("==============================")

# Unfreeze last 2 blocks of features for a quick fine-tune
# MobileNetV2 features is a Sequential of blocks. We'll unfreeze the last few.
for i, block in enumerate(model_tl.features):
    if i >= len(model_tl.features) - 3:   # last 3 blocks
        for p in block.parameters():
            p.requires_grad = True

print("Trainable params (fine-tune):", count_trainable_params(model_tl))


B) Fine-tuning: Unfreeze last conv blocks + classifier
Trainable params (fine-tune): 1218890


In [15]:
# Lower LR for fine-tuning
optimizer_ft = optim.Adam(filter(lambda p: p.requires_grad, model_tl.parameters()), lr=1e-4)

hist_ft, time_ft = train_model(
    model_tl, train_loader, val_loader,
    criterion, optimizer_ft,
    epochs=EPOCHS_FT,
    label="[FineTune] "
)
print(f"[FineTune] Training time: {time_ft:.1f}s")

[FineTune] Epoch 01/2 | train acc 0.785 loss 0.6746 | val acc 0.785 loss 0.6567
[FineTune] Epoch 02/2 | train acc 0.824 loss 0.5305 | val acc 0.803 loss 0.5780
[FineTune] Training time: 204.3s


In [16]:
final_tl_val_acc = hist_ft[-1][3] if len(hist_ft) > 0 else hist_tl[-1][3]

In [17]:
# -----------------------------
# 8) Task C: Train from Scratch (MobileNetV2 weights=None)
# -----------------------------
print("\n==============================")
print("C) Train From Scratch: MobileNetV2 (no pretrained weights)")
print("==============================")

model_scratch = models.mobilenet_v2(weights=None)
model_scratch.classifier[1] = nn.Linear(model_scratch.classifier[1].in_features, num_classes)
model_scratch = model_scratch.to(DEVICE)

print("Trainable params (scratch):", count_trainable_params(model_scratch))


C) Train From Scratch: MobileNetV2 (no pretrained weights)
Trainable params (scratch): 2236682


In [18]:
criterion_s = nn.CrossEntropyLoss()
optimizer_s = optim.Adam(model_scratch.parameters(), lr=1e-3)

hist_s, time_s = train_model(
    model_scratch, train_loader, val_loader,
    criterion_s, optimizer_s,
    epochs=EPOCHS_SCRATCH,
    label="[Scratch] "
)
print(f"[Scratch] Training time: {time_s:.1f}s")

final_s_val_acc = hist_s[-1][3]

[Scratch] Epoch 01/2 | train acc 0.272 loss 1.9571 | val acc 0.101 loss 2.7695
[Scratch] Epoch 02/2 | train acc 0.365 loss 1.6839 | val acc 0.367 loss 1.7661
[Scratch] Training time: 396.4s


In [19]:
# -----------------------------
# 9) Summary table
# -----------------------------
# TL head-only final val acc:
final_head_val_acc = hist_tl[-1][3]
total_tl_time = time_tl + time_ft

print("\n==============================")
print(" Summary ( comparison)")
print("==============================")
print(f"Transfer Learning (Head only)  | Train Acc: {hist_tl[-1][1]:.3f}  | Val Acc: {final_head_val_acc:.3f} | Time: {time_tl:.1f}s")
print(f"Transfer Learning + Fine-tune   | Train Acc: {hist_ft[-1][1]:.3f} | Val Acc: {final_tl_val_acc:.3f} | Time: {total_tl_time:.1f}s")
print(f"Train from Scratch              | Train Acc: {hist_s[-1][1]:.3f} | Val Acc: {final_s_val_acc:.3f} | Time: {time_s:.1f}s")


 Summary ( comparison)
Transfer Learning (Head only)  | Train Acc: 0.744  | Val Acc: 0.756 | Time: 203.7s
Transfer Learning + Fine-tune   | Train Acc: 0.824 | Val Acc: 0.803 | Time: 408.0s
Train from Scratch              | Train Acc: 0.365 | Val Acc: 0.367 | Time: 396.4s


In [20]:
def build_efficitnet_transfer(num_classes=10):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

    # Freeze feature extractor
    for p in model.features.parameters():
        p.requires_grad = False

    # Replace classifier
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model

In [21]:
model_new_tl=build_efficitnet_transfer(num_classes).to(DEVICE)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\liyoubao/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 47.0MB/s]


In [22]:
model_scratch = models.efficientnet_bo(weights=None)
model_scratch.classifier[1] = nn.Linear(model_scratch.classifier[1].in_features,num_classes)

AttributeError: module 'torchvision.models' has no attribute 'efficientnet_bo'